# INR Tokenizer Verification

1. **Load:** Import a raw `.pth` file from the dataset.

In [1]:
import torch
import os
import sys

sys.path.append(os.path.abspath('../')) 

from src.tokenizer import INRTokenizer

# Initialize
tokenizer = INRTokenizer()
print("Tokenizer Loaded Successfully")

Tokenizer Loaded Successfully


2. **Tokenize:** Convert the `state_dict` into the nested list structure.

In [2]:
path = "./data/mnist_png_training_0_1/checkpoints/model_final.pth"
state_dict = torch.load(path, map_location='cpu')
tokens = tokenizer.tokenize(state_dict)

print(f"Number of layers tokenized: {len(tokens)}\n")
for i, layer in enumerate(tokens):
    print(f"Layer {i} contains {len(layer)} neuron tokens.\n")
    print(f"Neuron shape from this layer: {layer[0].shape}\n")
    #print(f"Layer list {layer}\n")

Number of layers tokenized: 3

Layer 0 contains 32 neuron tokens.

Neuron shape from this layer: torch.Size([3])

Layer 1 contains 32 neuron tokens.

Neuron shape from this layer: torch.Size([33])

Layer 2 contains 1 neuron tokens.

Neuron shape from this layer: torch.Size([33])



3. **Detokenize:** Reconstruct the `state_dict`.
4. **Check:** Use `torch.equal` to ensure the original and reconstructed tensors are identical.

In [3]:
reconstructed_dict = tokenizer.detokenize(tokens)
for key in state_dict.keys():
    if torch.equal(state_dict[key], reconstructed_dict[key]):
        print(f"✅ {key} is an exact bit-match.")
    else:
        print(f"❌ {key} has changed.")

✅ seq.0.weight is an exact bit-match.
✅ seq.0.bias is an exact bit-match.
✅ seq.1.weight is an exact bit-match.
✅ seq.1.bias is an exact bit-match.
✅ seq.2.weight is an exact bit-match.
✅ seq.2.bias is an exact bit-match.


## Discretizer Test

5. **Discretize:** Quantize the tokens into `n` bins and inspect before/after values.
6. **Rebuild:** Detokenize the quantized tokens back into a `state_dict`.

In [4]:
from src.discretizer import discretize

N_BINS = 100

q_tokens = discretize(tokens, n_bins=N_BINS)

print(f"Discretized with n_bins={N_BINS}\n")
print("--- Before vs After (Layer 0, first 3 neurons, full token) ---")
for i in range(3):
    orig = tokens[0][i]
    quant = q_tokens[0][i]
    print(f"\nNeuron {i}:")
    print(f"  original : {orig.tolist()}")
    print(f"  quantized: {quant.tolist()}")

Discretized with n_bins=100

--- Before vs After (Layer 0, first 3 neurons, full token) ---

Neuron 0:
  original : [0.06459876149892807, 0.030936982482671738, -0.005675104912370443]
  quantized: [0.06565654277801514, 0.03535354137420654, -0.0050505101680755615]

Neuron 1:
  original : [-0.02965756133198738, 0.010647762566804886, 0.01125216856598854]
  quantized: [-0.02525252103805542, 0.015151500701904297, 0.015151500701904297]

Neuron 2:
  original : [0.020368244498968124, 0.07542167603969574, 0.03566013276576996]
  quantized: [0.02525252103805542, 0.07575756311416626, 0.03535354137420654]


In [5]:
q_state_dict = tokenizer.detokenize(q_tokens)

print("--- Quantized state_dict vs original ---")
for key in state_dict.keys():
    orig_w = state_dict[key]
    quant_w = q_state_dict[key]
    max_diff = (orig_w - quant_w).abs().max().item()
    mean_diff = (orig_w - quant_w).abs().mean().item()
    exact = torch.equal(orig_w, quant_w)
    status = "exact match" if exact else f"max_diff={max_diff:.6f}, mean_diff={mean_diff:.6f}"
    print(f"  {key}: {status}")

--- Quantized state_dict vs original ---
  seq.0.weight: max_diff=0.004884, mean_diff=0.001995
  seq.0.bias: max_diff=0.004726, mean_diff=0.002461
  seq.1.weight: max_diff=0.005028, mean_diff=0.002603
  seq.1.bias: max_diff=0.005004, mean_diff=0.002790
  seq.2.weight: max_diff=0.004880, mean_diff=0.002446
  seq.2.bias: max_diff=0.004535, mean_diff=0.004535
